# Calling gene hits in a pooled screen

A pooled optical CRISPR screen targets each gene with many single guides, read out one cell at a time. A per-guide phenotype is noisy and a few guides of any gene are duds, so a gene call has to pool a gene's guides rather than trust any one of them. The question each gene poses is whether its guides, taken together, move further than non-targeting guides do.

This case study scores the guides of the `cp_posh` screen for phenotypic activity, then calls genes with {func}`~mantispy.tl.aggregate_guides`, which combines a gene's guides and calibrates the combined statistic against random same-size groups of non-targeting-control guides. The screen carries two control classes, non-targeting and intergenic, so one can set the per-guide activity scale while the other is the null the gene call is judged against.

In [ ]:
import numpy as np

import mantispy as mt

guides = mt.ds.cp_posh(aggregated=True)
print(guides)
guides.obs["Metadata_Gene"].value_counts().head(4)

AnnData object with n_obs × n_vars = 1551 × 1278
    obs: 'Metadata_Gene', 'Metadata_sgRNA', 'Metadata_CellCount', 'Metadata_Perturbation', 'Metadata_Plate', 'Metadata_Control', 'Metadata_Perturbation_Type'
    var: 'object', 'feature_group', 'feature', 'channel', 'scale', 'angle', 'gray_levels', 'radial_bin', 'params', 'is_feature'
    uns: 'mantispy'
    layers: None (.X)


Metadata_Gene
nontargeting    200
intergenic      199
ACIN1            10
ACO2             10
Name: count, dtype: int64

One profile per guide, about ten guides per gene, and two control classes: `nontargeting` guides that carry a guide hitting nothing, and `intergenic` guides that cut a gene-free locus. Both should be phenotypically silent, which is what lets one play the part of the activity reference and the other the part of the null.

In [ ]:
# Per-guide activity: how far each guide sits from the intergenic control centre, in control units.
# Standardise every feature by the intergenic guides' median and MAD, then take the size of the
# standardised profile. The intergenic class sets the scale; it is not scored as a gene.
X = np.asarray(guides.X, dtype=np.float64)
intergenic = (guides.obs["Metadata_Gene"] == "intergenic").to_numpy()

med = np.median(X[intergenic], axis=0)
mad = np.median(np.abs(X[intergenic] - med), axis=0) * 1.4826
keep = mad > 1e-9
z = (X[:, keep] - med[keep]) / mad[keep]
activity = np.sqrt(np.mean(z**2, axis=1))

# One-sided p: the share of intergenic guides whose activity reaches this guide's, +1 smoothed.
ref = np.sort(activity[intergenic])
above = ref.size - np.searchsorted(ref, activity, side="left")
p = (above + 1) / (ref.size + 1)
p[intergenic] = np.nan  # the reference class, not a gene to test
guides.obs["activity_p"] = p
print(f"non-targeting guides for the null: {(guides.obs['Metadata_Gene'] == 'nontargeting').sum()}")

non-targeting guides for the null: 200


`activity_p` is small when a guide moves the cells away from the intergenic centre. {func}`~mantispy.tl.aggregate_guides` reads it, combines each gene's guides by Stouffer's z, and compares the combined value against random same-size groups of the non-targeting guides. Stouffer rewards a consistent shift across a gene's guides; a gene whose guides all move a little is called, while a gene carried by one lucky guide is not.

In [ ]:
mt.tl.aggregate_guides(
    guides,
    score="activity_p",
    guide="Metadata_sgRNA",
    gene="Metadata_Gene",
    control="nontargeting",
    method="stouffer",
    n_null=20000,
    alpha=0.05,
    seed=0,
)
table = guides.uns["mantispy"]["gene_aggregation"]
print(f"{int(table['is_hit'].sum())} of {len(table)} genes called at q < 0.05")
table.head(12)

119 of 124 genes called at q < 0.05


,gene,n_guides,statistic,pvalue,qvalue,is_hit
0,TPR,10,3.290871,0.00005,0.000067,True
1,TPT1,9,3.499075,0.00005,0.000067,True
2,TUBA1C,10,4.647507,0.00005,0.000067,True
3,TUBB,10,5.741675,0.00005,0.000067,True
4,TUBG1,10,4.970730,0.00005,0.000067,True
5,TUBGCP2,10,3.138307,0.00005,0.000067,True
6,TUBGCP3,10,6.311109,0.00005,0.000067,True
7,TUBGCP4,9,4.660966,0.00005,0.000067,True
8,TUBGCP5,10,3.822896,0.00005,0.000067,True
9,TUT1,9,5.006442,0.00005,0.000067,True


The top of the list is the cell's structural machinery: the gamma-tubulin ring complex that nucleates microtubules (`TUBG1`, `TUBGCP2` through `TUBGCP5`), the tubulins themselves (`TUBB`, `TUBA1C`), the nuclear-pore anchor `TPR`, and the AAA-ATPase `VCP`. Knocking any of these out derails the cell in a way a morphology readout cannot miss, so every one of their guides moves, and Stouffer stacks the evidence into a tiny p-value.

In [ ]:
# The genes that do not clear the cut, weakest last.
table.tail(6)

,gene,n_guides,statistic,pvalue,qvalue,is_hit
118,ACO2,10,0.822512,0.041598,0.043346,True
119,DDX11,10,0.424183,0.105495,0.109011,False
120,KIF18A,10,0.399453,0.110844,0.113593,False
121,CCDC115,10,0.389919,0.112794,0.114643,False
122,MBTPS1,10,-0.582145,0.486026,0.489977,False
123,MBTPS2,10,-0.871702,0.622469,0.622469,False


The genes that fall out are the ones a general morphology assay is blind to. `MBTPS1` and `MBTPS2` cut SREBP processing, a lipid-regulatory step with little shape consequence here, and their combined statistic is negative, below the non-targeting spread. The method is not simply calling everything: a hit-enriched screen leaves a handful of genes uncalled, in the right place.

In [ ]:
# Fisher instead of Stouffer: combine the guides' p-values, which fires on a single strong guide.
fisher = guides.copy()
mt.tl.aggregate_guides(
    fisher,
    score="activity_p",
    guide="Metadata_sgRNA",
    gene="Metadata_Gene",
    control="nontargeting",
    method="fisher",
    n_null=20000,
    alpha=0.05,
    seed=0,
)
ft = fisher.uns["mantispy"]["gene_aggregation"]
both = set(table.loc[table["is_hit"], "gene"]) & set(ft.loc[ft["is_hit"], "gene"])
print(f"stouffer hits: {int(table['is_hit'].sum())}, fisher hits: {int(ft['is_hit'].sum())}, shared: {len(both)}")

stouffer hits: 119, fisher hits: 119, shared: 119


Stouffer and Fisher agree exactly on this screen, because its hits are strong and every guide of a hit gene moves, so there is no lone-guide gene for Fisher to rescue or Stouffer to miss. The two part company on a gene with one potent guide among duds: Fisher calls it, Stouffer does not. Which is right depends on how much a single reagent is trusted, so the method leaves the choice to the caller.

## Summary

A pooled screen gives a noisy phenotype per guide, so a gene call pools the guides. {func}`~mantispy.tl.aggregate_guides` combines them against a non-targeting null matched to the gene's guide count, turning per-guide activity into a calibrated per-gene hit list. Here it recovers the microtubule and nuclear-pore machinery at the top and leaves the morphologically silent genes uncalled.